# FLAb 抗体亲和力预测 — 结果分析

本 notebook 用于分析 `train.py` 训练完成后的结果。

**前提**：`results/affinity_model/summary_all_losses.csv` 已生成。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import sys, os

# 确保在 FLAb/ 根目录下运行
sys.path.insert(0, '.')
from visualization.style import apply_style, COLORS, LOSS_COLORS, LOSS_LABELS
apply_style()

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

SUMMARY_PATH = 'results/affinity_model/summary_all_losses.csv'
print('加载结果...')
df = pd.read_csv(SUMMARY_PATH)
print(f'共 {len(df)} 条记录，{df["name"].nunique()} 个 benchmark，{df["loss"].nunique()} 种 loss')
df.head()

## 1. 总体指标：三种损失函数对比

In [ ]:
# 分组统计：每种 loss 的 Spearman 汇总
summary = df.groupby('loss')['spearman_test'].agg(['mean', 'median', 'std', 'min', 'max', 'count'])
summary.columns = ['均值', '中位数', '标准差', '最低', '最高', '有效数量']
summary = summary.round(4)
print('=== 消融实验汇总 ===')
display(summary)

## 2. 图1：消融实验箱线图

In [ ]:
from visualization.viz_stats import plot_ablation_boxplot

os.makedirs('figures', exist_ok=True)
plot_ablation_boxplot(df, save_path='figures/fig1_ablation_boxplot.pdf')

## 3. 图2：Benchmark × Loss 热力图

In [ ]:
from visualization.viz_stats import plot_heatmap

# top_n=30 展示样本量最多的30个benchmark
plot_heatmap(df, save_path='figures/fig2_heatmap.pdf', top_n=30)

## 4. 找出最优和最差的 Benchmark

In [ ]:
# 按 RankNet 的 Spearman 排序，找出最优和最差
ranknet_df = df[df['loss'] == 'ranknet'].sort_values('spearman_test', ascending=False)

print('=== Top 10 Benchmark（RankNet, Spearman 最高）===')
display(ranknet_df[['name', 'n', 'spearman_test', 'val_spearman']].head(10).round(4))

print('\n=== Bottom 10 Benchmark（RankNet, Spearman 最低）===')
display(ranknet_df[['name', 'n', 'spearman_test', 'val_spearman']].tail(10).round(4))

## 5. 三种 Loss 的逐 Benchmark 对比

In [ ]:
# pivot：每行是一个 benchmark，三列是三种 loss 的 Spearman
pivot = df.pivot(index='name', columns='loss', values='spearman_test')
pivot = pivot[['mse', 'hinge', 'ranknet']]

# RankNet 比 MSE 提升多少
pivot['ranknet_vs_mse'] = pivot['ranknet'] - pivot['mse']

print(f'RankNet 平均比 MSE 高: {pivot["ranknet_vs_mse"].mean():.4f}')
print(f'RankNet 在 {(pivot["ranknet_vs_mse"] > 0).sum()}/{len(pivot)} 个 benchmark 上优于 MSE')

# 绘制散点图：MSE vs RankNet
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for ax, (col_a, col_b) in zip(axes, [('mse', 'ranknet'), ('hinge', 'ranknet')]):
    x = pivot[col_a].dropna()
    y = pivot[col_b].reindex(x.index).dropna()
    x = x.reindex(y.index)

    ax.scatter(x, y, alpha=0.7, s=40,
               color=[LOSS_COLORS[col_b] if yv > xv else LOSS_COLORS[col_a]
                      for xv, yv in zip(x, y)])
    # 对角线：两者相等
    lims = [min(x.min(), y.min()) - 0.05, max(x.max(), y.max()) + 0.05]
    ax.plot(lims, lims, 'k--', alpha=0.3, linewidth=1)
    ax.set_xlabel(f'{LOSS_LABELS[col_a]} Spearman', fontsize=10)
    ax.set_ylabel(f'{LOSS_LABELS[col_b]} Spearman', fontsize=10)
    n_better = (y > x).sum()
    ax.set_title(f'{LOSS_LABELS[col_b]} 在 {n_better}/{len(x)} 个 benchmark 上更优', fontsize=10)
    ax.set_xlim(lims); ax.set_ylim(lims)

plt.suptitle('各 Benchmark 上不同损失函数的 Spearman 对比', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig3_loss_comparison_scatter.pdf')
plt.show()

## 6. 数据集大小 vs Spearman 关系

In [ ]:
# 观察：小数据集 Spearman 方差大，大数据集更稳定
fig, ax = plt.subplots(figsize=(8, 5))

for loss_name in ['mse', 'hinge', 'ranknet']:
    sub = df[df['loss'] == loss_name].dropna(subset=['spearman_test'])
    ax.scatter(sub['n'], sub['spearman_test'],
               label=LOSS_LABELS[loss_name],
               color=LOSS_COLORS[loss_name],
               alpha=0.6, s=35)

ax.axhline(0, color=COLORS['gray'], linewidth=1, linestyle='--', alpha=0.5)
ax.set_xscale('log')  # 用对数轴，数据集大小跨度大
ax.set_xlabel('数据集序列数（对数轴）', fontsize=11)
ax.set_ylabel('Spearman 相关系数（test）', fontsize=11)
ax.set_title('数据集规模 vs 预测效果', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('figures/fig4_size_vs_spearman.pdf')
plt.show()

## 7. 损失函数原理图（不依赖训练结果）

In [ ]:
from visualization.viz_loss import plot_loss_curves, plot_pairwise_concept

plot_loss_curves(save_path='figures/fig5_loss_curves.pdf')

In [ ]:
plot_pairwise_concept(save_path='figures/fig6_pairwise_concept.pdf')

## 8. 输出所有图片路径

In [ ]:
import glob
figures = sorted(glob.glob('figures/*.pdf'))
print(f'共生成 {len(figures)} 张图：')
for f in figures:
    size_kb = os.path.getsize(f) // 1024
    print(f'  {f}  ({size_kb} KB)')